<a href="https://colab.research.google.com/github/briskicedteaa/Modern-Computational-Biology/blob/main/Molecular_Dynamics_with_OpenMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openmm
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout

In [2]:
pdb = PDBFile('6LUQ.pdb')

In [3]:
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

In [4]:
!pip install pdbfixer

In [9]:
from pdbfixer import PDBFixer
from openmm.app import Modeller

fixer = PDBFixer(filename='6LUQ.pdb')
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.removeHeterogens(True)

modeller = Modeller(fixer.topology, fixer.positions)
modeller.addHydrogens(forcefield, pH=7.0)
fixer.topology = modeller.topology
fixer.positions = modeller.positions

system = forcefield.createSystem(fixer.topology, nonbondedMethod=PME,
        nonbondedCutoff=1*nanometer, constraints=HBonds)

In [13]:
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.004*picoseconds)
simulation = Simulation(fixer.topology, system, integrator)
simulation.context.setPositions(fixer.positions)

In [14]:
simulation.minimizeEnergy(maxIterations=100)

In [15]:
simulation.reporters.append(PDBReporter('output.pdb', 1000))

In [16]:
simulation.reporters.append(StateDataReporter(stdout, 100, step=True,
        potentialEnergy=True, temperature=True))

print("Running simulation...")
simulation.step(1000)

Running simulation...
#"Step","Potential Energy (kJ/mole)","Temperature (K)"
100,-26401.98655637187,186.26870068143145
200,-24103.397525324923,231.32228249219617
300,-22993.607155622187,255.3425472013097
400,-22359.10087896462,276.8220258746512
500,-21667.021653499447,279.0062840292428
600,-21736.000238848243,293.107910436144
700,-21209.852580927673,293.0955899040634
800,-21287.824278235617,297.5805650897231
900,-21633.3386506307,302.72300047002716
1000,-21674.283516276177,299.57263711102246
